<a href="https://colab.research.google.com/github/zhengyangsu/Natural-Language-Processing/blob/main/latin_translation_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Latin to English Translation with LoRA

Adapted from the Week 1 tutorial `movie_review_sentiment_classifier_with_bert.ipynb`  
Use a GPU runtime

# 1. Setup Python Libraries (pip)

In [1]:
# transformers = models and tokenizers, peft = LoRA, bitsandbytes = 4-bit loading,
# accelerate = GPU placement, datasets = data loading, --quiet hides the long install log
!pip install transformers peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 28.3 MB/s  0:00:01


# 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os

# Path to your NLP_Project folder in Google Drive
project_path = '/content/drive/MyDrive/NLP_Project'

# Create the folder if it does not exist
if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f"Created directory: {project_path}")

# Change the current working directory
os.chdir(project_path)
print(f"Current working directory set to: {os.getcwd()}")

Current working directory set to: /content/drive/MyDrive/NLP_Project


# 3. Hugging Face Auth

In [4]:
from huggingface_hub import notebook_login
from huggingface_hub import whoami

#notebook_login()
!hf auth login
!hf auth whoami
#print(whoami()['name'])

Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: PIRM-Y0VJ

    Waiting for authorization.......
Token is valid.
The token `oauth-zhengyangsu` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-zhengyangsu`
Note: This token will be refreshed automatically when it expires.
✓ Logged in
  user: zhengyangsu


# 4. Load the Dataset

In [11]:
from datasets import load_dataset
raw = load_dataset('grosenthal/latin_english_parallel')

## Data Cleanup

In [23]:
!pip install langid --quiet # langid to check latin in en key

import re, unicodedata
from langid.langid import LanguageIdentifier, model

ident = LanguageIdentifier.from_modelstring(model, norm_probs=True)

# Unicode normalize, remove html tags, collapse space
def clean_text(s):
    s = unicodedata.normalize('NFC', s)   # make accented letters use one consistent form
    s = re.sub(r'<[^>]+>', ' ', s)        # remove tags such as <br />
    return ' '.join(s.split())            # collapse whitespace, keep original case

# True if the en column is 0.9 confident detected as Latin
def en_is_latin(x):
    if len(x['en'].split()) < 6:          # skip short text
        return False
    lang, prob = ident.classify(x['en'])
    return lang == 'la' and prob > 0.9

# Keep pair only if it passes all checks
def is_ok(x):
    n_la, n_en = len(x['la'].split()), len(x['en'].split())
    if n_la < 3 or n_en < 3:              # sentence too short
        return False
    if en_is_latin(x):                    # en column is not English
        return False
    return 0.3 <= n_en / n_la <= 4        # lengths misalignment

# Apply text cleaning to every split
raw = raw.map(lambda x: {'la': clean_text(x['la']), 'en': clean_text(x['en'])})

# Filter splits with the same rule
for k in ['train', 'valid', 'test']:
    before = len(raw[k])
    raw[k] = raw[k].filter(is_ok)
    print(k, before, '->', len(raw[k]))

Map:   0%|          | 0/98960 [00:00<?, ? examples/s]

Map:   0%|          | 0/1008 [00:00<?, ? examples/s]

Map:   0%|          | 0/1012 [00:00<?, ? examples/s]

Filter:   0%|          | 0/98960 [00:00<?, ? examples/s]

train 98960 -> 98225


Filter:   0%|          | 0/1012 [00:00<?, ? examples/s]

valid 1012 -> 1002


Filter:   0%|          | 0/1008 [00:00<?, ? examples/s]

test 1008 -> 996


## Check for Overlapping

In [20]:
N_TRAIN = 2000
N_VAL   = 200
N_TEST  = 200
SEED    = 42

# Only used for comparing sentences, never changes the data
def key(s):
    return s.lower()

# 1. Check split overlapping
full_train_la = {key(x) for x in raw['train']['la']}
print('test/train overlap:', sum(key(x) in full_train_la for x in raw['test']['la']), '/', len(raw['test']))
print('valid/train overlap:', sum(key(x) in full_train_la for x in raw['valid']['la']), '/', len(raw['valid']))

# val/test set size
val = raw['valid'].select(range(N_VAL))
val_la = {key(x) for x in val['la']}
# take test rows in val
test = raw['test'].filter(lambda x: key(x['la']) not in val_la).select(range(N_TEST))

# remove val/test sentences from train
held_la = {key(x) for x in val['la']} | {key(x) for x in test['la']}
held_en = {key(x) for x in val['en']} | {key(x) for x in test['en']}

pool = raw['train'].filter(
    lambda x: key(x['la']) not in held_la and key(x['en']) not in held_en
)
print('train pool:', len(raw['train']), '->', len(pool))

# remove duplicate Latin sentences inside train
seen = set()
def first_time(x):
    k = key(x['la'])
    if k in seen:
        return False
    seen.add(k)
    return True
pool = pool.filter(first_time)
print('unique:', len(pool))

dataset = {
    'train': pool.shuffle(seed=SEED).select(range(N_TRAIN)),
    'val':   val,
    'test':  test,
}

# verification
train_la = {key(x) for x in dataset['train']['la']}
print('train ∩ test:', sum(key(x) in train_la for x in dataset['test']['la']))
print('train ∩ val :', sum(key(x) in train_la for x in dataset['val']['la']))
print('val ∩ test  :', len({key(x) for x in dataset['val']['la']} & {key(x) for x in dataset['test']['la']}))

test/train overlap: 7 / 1008
valid/train overlap: 6 / 1012


Filter:   0%|          | 0/1008 [00:00<?, ? examples/s]

Filter:   0%|          | 0/98960 [00:00<?, ? examples/s]

train pool: 98960 -> 98904


Filter:   0%|          | 0/98904 [00:00<?, ? examples/s]

unique: 98617
train ∩ test: 0
train ∩ val : 0
val ∩ test  : 0


##Dataframe Display

In [21]:
import pandas as pd

print({k: len(v) for k, v in raw.items()})
print(dataset['train'].column_names)

pd.set_option('display.max_colwidth', None)

dataset['train'].to_pandas().head()

from IPython.display import display

for name in ['train', 'val', 'test']:
    print(name)
    display(dataset[name].to_pandas().head(3))

{'train': 98960, 'test': 1008, 'valid': 1012}
['id', 'la', 'en', 'file']
train


,id,la,en,file
0,79378,quid facietis in die visitationis et calamitatis de longe venientis ad cuius fugietis auxilium et ubi derelinquetis gloriam vestram,"What will you do in the day of visitation, and of the calamity which cometh from afar? to whom will ye flee for help? and where will ye leave your glory?",final_alignments\Vulgate_Bible.json
1,26250,"Ridicule magis hoc dictum quam vere aestimo; quando et formosos saepe inveni pessimos, et turpi facie multos cognovi optimos.","This, I suppose, has been recounted more in pursuit of amusement rather than consideration for veracity; for I have frequently encountered attractive individuals who were wicked, and have been acquainted with numerous individuals possessing unattractive countenances who were exemplars of virtue.",final_alignments\Phaedrus_Fables_Book3.json
2,33373,Contra Alexander expedito ac levi agmine strenue invectus est.,"On the other hand, Alexander vigorously attacked them with his force, unburdened and lightly armed.",final_alignments\Quintus_Curtius_Alexander_Book8.json


val


,id,la,en,file
0,36149,"nam memor originis tuae nobilium sibi persuasit universitas, quamdiu nos Sabini familia rexerit, Sabiniani familiam non timendam.","For the entire aristocracy, who recall the lineage that you belong to, are certain that as long as the descendants of Sabinus are our leaders, there is no reason for us to be afraid of the descendants of Sabinianus.",final_alignments\Sidonius_Letters_Book3.json
1,81250,super aspidem et basiliscum calcabis conculcabis leonem et draconem,super aspidem et basiliscum ambulabis et; conculcabis leonem et draconem,final_alignments\Vulgate_Bible.json
2,27252,"Tanta tibi bene faciendi vis, ut indulgentia tua necessitates aemuletur.",Such is your ability to perform well that your merciful kindness has an equivalent impact as harsh compulsion.,final_alignments\Pliny_Younger_Panegyricus.json


test


,id,la,en,file
0,32668,"Tibi autem, qui sapis, quam potest denuntio ipsi mihi indices quid ex eis cognoveris, ne possis infitiari dixisse, quae dixeris.","But I offer you a serious caution that you tell me directly what you have gained from those sacrifices, since I know, so that you will not be able to deny having spoken to me.",final_alignments\Quintus_Curtius_Alexander_Book7.json
1,90310,currens autem vir de Beniamin ex acie venit in Silo in die illo scissa veste et conspersus pulvere caput,"And there ran a man of Benjamin out of the army, and came to Silo the same day, with his clothes rent, and his head strewed with dust.",final_alignments\Vulgate_Bible.json
2,16191,"Exposuit quid iniquitas loci posset, quid ipse ad Avaricum sensisset, cum sine duce et sine equitatu deprehensis hostibus exploratam victoriam dimisisset, ne parvum modo detrimentum in contentione propter iniquitatem loci accideret.","""And he said what disadvantage the ground could have on the outcome, recalling his own opinion at Avaricum where, despite having surprised the enemy without the aid of a general or cavalry, he abandoned a certain victory in order to avoid even a small loss caused by the disadvantage of the position.""",final_alignments\Caesar_DBG_Book7.json


# Tokenize the Dataset

In [ ]:
from transformers import AutoTokenizer

checkpoint = 'hathibelagal/llama-3.2-latin'  #base model
tokenizer = AutoTokenizer.from_pretrained(checkpoint, clean_up_tokenization_spaces=False)  #tokenizer that matches the model's vocabulary
tokenizer.pad_token = tokenizer.eos_token  # Llama has no padding token, so reuse the end-of-text token

def tokenization(batch):  # function applied to a batch of rows
    # join each Latin sentence with its English translation into one prompt-style text
    texts = [f"Latin: {la}\nEnglish: {en}" for la, en in zip(batch['la'], batch['en'])]
    return tokenizer(texts, truncation=True, max_length=256)  # text -> token ids; cap to 256 tokens

tokenized = {k: v.map(tokenization, batched=True, remove_columns=v.column_names)  # run it on each split in batches, drop raw text columns
             for k, v in dataset.items()}  # loop over train / val / test

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

# Load the Model and Add LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True,  # store the frozen base weights in 4 bits
                         bnb_4bit_quant_type='nf4',  # NF4 is the 4-bit format designed for QLoRA
                         bnb_4bit_compute_dtype=torch.float16)  # compute in 16-bit floats

model = AutoModelForCausalLM.from_pretrained(checkpoint,  # download base model
                                             quantization_config=bnb,  # apply the 4-bit settings above
                                             device_map='auto')  # training on the GPU automatically

model = prepare_model_for_kbit_training(
    model,
    gradient_checkpointing_kwargs={'use_reentrant': False})  # casts some layers to full precision, cant update 4bit weight

model = get_peft_model(model, LoraConfig(r=16,  # rank of the LoRA matrices: higher = more capacity
                                         lora_alpha=32,  # scaling factor for the LoRA update
                                         target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # add LoRA to the attention layers
                                         task_type='CAUSAL_LM'))  # word-prediction with Parameter Efficient Fine Tuning（
model.print_trainable_parameters()  # show how few parameters are actually trained

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848


# Setup Training Configurations

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling  # training tools

training_args = TrainingArguments(
    seed=42,  # fixed random seed
    output_dir='./results',  # folder for chkpts
    num_train_epochs=1,  # total passes
    per_device_train_batch_size=4,  # 4 examples per step (small to fit GPU memory)
    gradient_accumulation_steps=4,  # accumulate 4 steps before each update = effective batch size 16
    learning_rate=2e-4,  # step size
    fp16=True,  # 16-bit training (T4 supports fp16 not bf16)
    eval_strategy='epoch',  # evaluate on the validation set at the end of each epoch
    save_strategy='epoch',
    load_best_model_at_end=True,   # load best chkpt
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=10,
    report_to='none',  # do not log
)

trainer = Trainer(
    model=model,  # the LoRA model to train
    args=training_args,  # hyperparameter settings above
    train_dataset=tokenized['train'],  # data to learn from
    eval_dataset=tokenized['val'],  # data used for evaluation during training
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),  # builds padded batches and labels; mlm=False = predict the next word
)

# Evaluate the Un-Fine-Tuned Model for a Baseline

In [ ]:
import numpy as np  # for exp()

result = trainer.evaluate(tokenized['test'])  # loss on the test set (untrain LoRA starts at 0)
print(result, 'perplexity =', np.exp(result['eval_loss']))  # perplexity = exp(loss): lower = less random text

Training Loss,Validation Loss,Epoch
No log,3.219894,0


{'eval_loss': 3.2198944091796875} perplexity = 25.0254775811166


# Fine-Tune with LoRA

In [ ]:
trainer.train()  # run the LoRA fine tuning loop, Traning loss loging step is 500

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,2.793176


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=125, training_loss=2.798360107421875, metrics={'train_runtime': 589.5015, 'train_samples_per_second': 3.393, 'train_steps_per_second': 0.212, 'total_flos': 4999518002601984.0, 'train_loss': 2.798360107421875, 'epoch': 1.0})

# Evaluate After Fine-Tuning

In [ ]:
result = trainer.evaluate(tokenized['test'])  # same test set as the baseline, so the numbers are comparable
print(result, 'perplexity =', np.exp(result['eval_loss']))  # compare with the baseline perplexity

Training Loss,Validation Loss,Epoch
No log,2.794720,1


{'eval_loss': 2.794720411300659} perplexity = 16.358054586538767


# Try out some examples and save the adapter

In [22]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def translate(la, beams=1, max_new_tokens=120):
    out = pipe(f"Latin: {la}\nEnglish:",
               max_new_tokens=max_new_tokens,
               do_sample=False,
               num_beams=beams,                  # 1 = greedy，4 = beam search
               repetition_penalty=1.1 if beams > 1 else 1.0,
               pad_token_id=tokenizer.eos_token_id,   # avoid "Setting pad_token_id" warning
               return_full_text=False)
    return out[0]['generated_text'].split('\n')[0].strip()

q = "Quo usque tandem abutere, Catilina, patientia nostra?"
print('greedy:', translate(q))
print('beam=4:', translate(q, beams=4))

model.save_pretrained('latin_lora')  # save LoRA adapter

NameError: name 'tokenizer' is not defined

# Reuse LORA

In [ ]:
'''from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(checkpoint, quantization_config=bnb, device_map='auto')
model = PeftModel.from_pretrained(base, 'latin_lora')'''

#Evaluate custom test set

In [ ]:
#Custom test set from Perseus https://github.com/PerseusDL/canonical-latinLit
from datasets import load_dataset

mine = load_dataset('json', data_files={'test': 'my_test.jsonl'})['test']
mine = mine.select_columns(['la', 'en', 'file'])